In [8]:
import math
import pandas as pd
from collections import deque
from itertools import combinations, product
from typing import Union, List

In [9]:
def toStr(vec) -> str:
    return ''.join(map(str, vec))

def countOnes(vec) -> int:
    return vec.count(1)

def toInt(vec) -> int:
    return int("".join(str(i) for i in vec), 2)

def toBinVec(x, l=0) -> list:
    result = [int(i) for i in bin(x)[2:]]
    missing_zeros = l-len(result)
    if (missing_zeros > 0):
        result = [0] * missing_zeros + result
    return result

def binary_permutations(lst: List[int]) -> List[List[int]]:
    result = []
    for comb in combinations(range(len(lst)), lst.count(1)):
        permutation = [0] * len(lst)
        for i in comb:
            permutation[i] = 1
        result.append(permutation)
    return result

def find_min_max(vec) -> Union[int,int,int,list,list]:
    L = len(vec)
    vecToInt = toInt(vec)
    min = vecToInt
    max = 0
    odd_max = 0
    vec_min = vec
    vec_max = vec
    shifted = vec.copy()
    for i in range(L):
        if vecToInt < min:
            min = vecToInt
            vec_min = shifted
        if vecToInt > max:
            max = vecToInt
            vec_max = shifted
        temp = deque(shifted)
        temp.rotate(-1)
        shifted = list(temp)
        vecToInt = toInt(shifted)
    return min, max, odd_max, vec_min, vec_max

def find_rotational_distance(vec1, vec2) -> Union[int,int]:
    L = len(vec1)
    vec2ToInt = toInt(vec2)
    left_shifts = 0
    shifted = vec1.copy()
    for i in range(L):
        if toInt(shifted) == vec2ToInt:
            left_shifts = i
            break
        temp = deque(shifted)
        temp.rotate(-1)
        shifted = list(temp)
    return left_shifts, L-left_shifts

def calculate_c(k: int, l: int, N1: int) -> int:
    return 2**l-k**N1

def calculate_z(k: int, parity_vector: List[int]) -> int:
    indices = []
    for i, entry in enumerate(parity_vector, start=0):
        if entry == 1:
            indices.append(i)
    result = 0
    N1 = len(indices)
    for i, entry in enumerate(indices, start=0):
        result = result + k**(N1-(i+1)) * 2**(entry)
    return result

def cycle(k, z, c) -> list:
    result = [z]
    item = z
    while (next_item := (k*item+c)//2 if item % 2 == 1 else item // 2) not in result:
        item = next_item
        result.append(item)
    return result

def calculate_M(k: int, n: int, U: int) -> Union[int,pd.DataFrame]:
    if n == 0 or U == 0 or n < U:
        return 0, None
    df = pd.DataFrame(columns = ['parity_vector', 'shifted orbits', 'z vals', 'min_z'])
    p_v = [0] * n
    for i in range(U):
        p_v[i] = 1
    permutations = binary_permutations(p_v)
    for i in range(len(permutations)):
        vec = permutations[i]
        z_vals = []
        shifted_vecs = []
        shifted = vec.copy()
        for j in range(len(shifted)):
            shifted_vecs.append(''.join(map(str, shifted)))
            z_vals.append(calculate_z(k, shifted))
            temp = deque(shifted)
            temp.rotate(-1)
            shifted = list(temp)
        min_z_vals = min(z_vals) if len(z_vals) > 0 else 0
        df.loc[i] = [''.join(map(str, vec)), shifted_vecs, z_vals, min_z_vals]
    return df['min_z'].max(), df

def calculate_N(k:int, M:int, n:int, U:int) -> int:
    if M == 0:
        return 0
    r = math.gcd(n, U)
    sum = 0
    for i in range(r):
        sum += 2**(i*n/r)*k**(U-1-i*U/r)
    return 2*M - int(sum)

def generate_df(vecs: List[List]) -> pd.DataFrame:
    df = pd.DataFrame(columns = ['vec', 'L (len)', 'N (1s)', 'min', 'max', 'odd_max', 'vec_min', 'vec_max', 'rot_left', 'quot=(N*rot+1)/L'])
    for i in range(len(vecs)):
        vec = vecs[i]
        L = len(vec)
        min, max, odd_max, vec_min, vec_max = find_min_max(vec)
        rot_dist_left, rot_dist_right = find_rotational_distance(vec_min, vec_max)
        quot = (countOnes(vec)*rot_dist_left+1)/L
        df.loc[i] = [toStr(vec), L, countOnes(vec), min, max, odd_max, toStr(vec_min), toStr(vec_max), rot_dist_left, quot]
    return df

n=14
U=8
k=3
M, df_M = calculate_M(k, n, U)
N = calculate_N(k, M, n, U)
c = calculate_c(k, n, U)
print("M({0},{1})={2}".format(n, U, M))
print("N({0},{1})={2}".format(n, U, N))
print("c={0}".format(c))

M(14,8)=21109
N(14,8)=36575
c=9823


In [10]:
for i in range(0, 8+1):
    print(toStr(toBinVec(13, i)))

1101
1101
1101
1101
1101
01101
001101
0001101
00001101


In [11]:
#generate some cycles
rot_cases = []
max_values = []
k = 5
for i in range(1, 80000):
    vec = toBinVec(i)
    x_min, x_max, odd_x_max, vec_min, vec_max = find_min_max(vec)
    rot_dist_left, rot_dist_right = find_rotational_distance(vec_min, vec_max)
    l = len(vec_max)
    N1 = countOnes(vec)
    c = calculate_c(k, l, N1)
    quot = (N1*rot_dist_left+1)/l
    if quot.is_integer() and x_max not in max_values:
        max_values.append(x_max)
        rot_cases.append([x_min, x_max, toStr(vec_min), toStr(vec_max), l, rot_dist_left, N1, quot, cycle(k, calculate_z(k, vec_max), c), c])

df = pd.DataFrame(rot_cases, columns = ['min', 'max', 'B_min', 'B_max', 'l', 'r', 'N1', '(N1*r+1)/l', 'cycle', 'c'])


In [12]:
pd.set_option('display.expand_frame_repr', False)
print(df.loc[(df['c']>0) & (df['c']<1000)])
#print(df.loc[df['c'].isin([1,3,5,7,9,11,13,15])])
#print(df.loc[df['c'].isin([1])])

    min   max         B_min         B_max   l  r  N1  (N1*r+1)/l                                              cycle    c
2     1     4           001           100   3  2   1         1.0                                          [1, 4, 2]    3
4     1     8          0001          1000   4  3   1         1.0                                       [1, 8, 4, 2]   11
6     1    16         00001         10000   5  4   1         1.0                                   [1, 16, 8, 4, 2]   27
7     5    20         00101         10100   5  2   2         1.0                                [9, 26, 13, 36, 18]    7
10    1    32        000001        100000   6  5   1         1.0                               [1, 32, 16, 8, 4, 2]   59
12    1    64       0000001       1000000   7  6   1         1.0                           [1, 64, 32, 16, 8, 4, 2]  123
13    9    72       0001001       1001000   7  3   2         1.0                      [13, 84, 42, 21, 104, 52, 26]  103
14   21    84       0010101     

In [15]:
def exact_divisibility_two(x):
    d = 0
    while x%2 == 0:
        x //= 2
        d+=1
    return d

def generate_sequence(k, c, v1, n):
    collatz_set = []
    if n<1 or v1%2==0:
        return collatz_set
    vi = v1
    collatz_set.append(vi)
    for i in range(n-1):
        vi = k*vi+c
        while vi%2 == 0:
            vi //= 2**(exact_divisibility_two(vi))
            collatz_set.append(int(vi))
    return collatz_set

k = 5
c = 971
v1 = 1031
n = 7
print(generate_sequence(k, c, v1, n))
print(calculate_z(5,[1,1,1,1,1,0,0,0,0,0,0,0]))
print(calculate_z(5,[1,1,1,1,0,0,0,0,0,0,0,1]))
print(calculate_z(5,[0,0,0,0,0,0,0,1,1,1,1,1]))

k = 5
c = 1
v1 = 13
n = 5
print(generate_sequence(k, c, v1, n))

[1031, 3063, 8143, 20843, 52593, 1031, 3063]
1031
3063
131968
[13, 33, 83, 13, 33]


In [14]:
def z_inverse(k, z, l) -> int:
    i = 1
    while True:
        vec = toBinVec(i, l)
        z_test = calculate_z(k, vec)
        if z_test == z and len(vec) == l:
            return i
        if z_test > 99999999:
            #give up
            return 0
        i += 1

z_inv_min = z_inverse(3, 842, 8)
print(toStr(toBinVec(z_inv_min)))
z_inv_max = z_inverse(3, 319, 8)
print(toStr(toBinVec(z_inv_max)))

z_inv_max = z_inverse(5, 13, 4)
print(toStr(toBinVec(z_inv_max)))
z_inv_min = z_inverse(5, 83, 8)
print(toStr(toBinVec(z_inv_min)))

lst = list(map(list, product([0, 1], repeat=10)))
for candidate in lst:
    z_test = calculate_z(5, candidate)
    if z_test > 70 and z_test < 150:
        print("{0}, {1}".format(candidate, z_test))

print(calculate_z(5, [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]))
print(calculate_z(5, [1, 0, 0, 0, 0, 0, 0, 1, 0, 0]))


1011011
11011010
1001
0
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0], 128
[0, 0, 0, 0, 1, 0, 1, 0, 0, 0], 144
[0, 0, 0, 0, 1, 1, 0, 0, 0, 0], 112
[0, 0, 0, 1, 0, 0, 1, 0, 0, 0], 104
[0, 0, 0, 1, 0, 1, 0, 0, 0, 0], 72
[0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 148
[0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 84
[0, 1, 0, 0, 0, 0, 0, 1, 0, 0], 138
[0, 1, 0, 0, 0, 0, 1, 0, 0, 0], 74
[0, 1, 0, 1, 0, 1, 0, 0, 0, 0], 122
[0, 1, 0, 1, 1, 0, 0, 0, 0, 0], 106
[0, 1, 1, 0, 0, 0, 1, 0, 0, 0], 134
[0, 1, 1, 0, 0, 1, 0, 0, 0, 0], 102
[0, 1, 1, 0, 1, 0, 0, 0, 0, 0], 86
[0, 1, 1, 1, 0, 0, 0, 0, 0, 0], 78
[1, 0, 0, 0, 0, 0, 0, 1, 0, 0], 133
[1, 0, 0, 0, 1, 1, 0, 0, 0, 0], 137
[1, 0, 0, 1, 0, 0, 1, 0, 0, 0], 129
[1, 0, 0, 1, 0, 1, 0, 0, 0, 0], 97
[1, 0, 0, 1, 1, 0, 0, 0, 0, 0], 81
[1, 0, 1, 0, 0, 0, 1, 0, 0, 0], 109
[1, 0, 1, 0, 0, 1, 0, 0, 0, 0], 77
[1, 1, 0, 0, 0, 0, 1, 0, 0, 0], 99
13
133
